# hai07 - prapelatihan 540M dari nol (Kaggle GPU T4 x2, DDP, fp16, lanjut antar sesi 12 jam)

**Perkiraan jujur (terukur di bench T4x2)**: ±3.550 token/detik, ±148 detik per langkah (524 ribu token). Satu sesi ±11 jam = ±250 langkah (±130 juta token).
`MAX_STEPS = 20000` (±10,5 miliar token) butuh **±80 sesi** dan ±820 jam GPU (±28 minggu pada kuota 30 jam/minggu; kuota perlu dicek di akun Anda).
Mau lebih pendek? Ubah `MAX_STEPS` **sebelum sesi pertama** lalu jangan diubah lagi (jadwal LR cosine bergantung nilainya): 5000 langkah = ±2,6 miliar token = ±20 sesi.

**Dataset yang harus ada di Input (Add Input)**
1. `hai07-pretrain-kit`: unggah `kaggle/hai07-pretrain-kit.zip` (dibuat `python tools/package_pretrain_kit.py`, ±0,8 MB).
2. Shard data: tiap shard = dua berkas `000_0000N.u16` dan `000_0000N.u16.docs.npy` dari folder `data/scale/` (total ±26 GB; unggah sebagai beberapa dataset, mis. `hai07-scale-0` ... `hai07-scale-3`, masing-masing ±5-7 GB). **Pasang SEMUA shard yang sama di setiap sesi**; urutan data tersimpan di checkpoint dan ukuran data harus sama.
3. Sesi ke-2 dan seterusnya: dataset checkpoint sesi sebelumnya (mis. `hai07-ckpt540`; hanya berkas `gpt_last.pt` dan `gpt_last.meta.json` yang dipakai).

**Settings**: Accelerator = GPU T4 x2, Persistence = Files, Internet tidak perlu. Jalankan `SMOKE = True` dulu (±10 menit), lalu `False` dengan **Save & Run All (Commit)** supaya berjalan penuh tanpa Anda menunggu.

**Setelah tiap sesi selesai**: buka versi notebook > tab Output > buat/perbarui dataset `hai07-ckpt540` dari output (6,5 GB; berisi `gpt_last.pt`). Di sesi berikutnya pasang dataset itu (versi terbaru), ubah `FIRST_SESSION = False`.
Notebook berhenti sendiri dengan rapi (menulis `gpt_last.pt`) pada `TIME_LIMIT_S`, jadi tidak ada langkah yang hilang. Checkpoint juga ditulis tiap 50 langkah (±2 jam) sebagai cadangan bila sesi terputus.

**Peringatan**: kode DDP sudah terbukti di dua T4 untuk bench dan di CPU untuk alur latih (simpan, lanjut, batas waktu), tetapi **belum pernah melatih 540M sungguhan**. fp16 (T4 tanpa bf16) dapat membuat loss tidak stabil; pantau `loss` dan `skipped_non_finite` di log.

In [ ]:
SMOKE = True            # True: uji 3 langkah (cek data, DDP, kecepatan, simpan); False: latih sungguhan
FIRST_SESSION = True    # True hanya di sesi pertama; di sesi berikutnya False (jika checkpoint tidak ditemukan, notebook berhenti, bukan mulai dari nol)
MAX_STEPS = 20000       # total langkah optimizer; JANGAN diubah setelah sesi pertama (jadwal cosine bergantung nilainya)
VAL_DOCS = 2000         # dokumen terakhir shard terakhir disisihkan untuk validasi (tidak dipakai melatih)
EXPECTED_SHARDS = 4     # jumlah shard data yang harus terpasang (kurangi bila hanya memakai sebagian)
TIME_LIMIT_S = 11 * 3600        # berhenti rapi + simpan gpt_last.pt setelah waktu ini (sesi Kaggle maksimal 12 jam)
HARD_KILL_S = TIME_LIMIT_S + 2400   # pengaman terakhir bila proses tersangkut
SAVE_EVERY = 50


In [ ]:
import glob, json, os, re, shutil, subprocess, sys, threading, time, zipfile
import numpy as np
WORK = "/kaggle/working/hai07-540m"; os.makedirs(WORK, exist_ok=True)
subprocess.run(["nvidia-smi", "-L"])
src_dir = next((os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/tools/train_gptV2.py", recursive=True)), None)
if src_dir:
    subprocess.run(["cp", "-r", src_dir + "/.", WORK], check=True)
else:
    z = glob.glob("/kaggle/input/**/hai07-pretrain-kit*.zip", recursive=True); assert z, "tambahkan dataset hai07-pretrain-kit sebagai Input"
    zipfile.ZipFile(z[0]).extractall(WORK)
os.chdir(WORK)
assert os.path.exists("tools/train_gptV2.py") and os.path.exists("data/prod/vocab_prod.json"), "kit tidak lengkap: bangun ulang dengan tools/package_pretrain_kit.py"
src = open("tools/train_gptV2.py", encoding="utf-8").read()
assert "--time-limit-s" in src and "WORLD_SIZE" in src, "kit lama: unggah hai07-pretrain-kit.zip terbaru"
import torch
n_gpu = torch.cuda.device_count()
print("torch", torch.__version__, "| GPU:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu >= 2, "pilih Accelerator GPU T4 x2"

shards = sorted(glob.glob("/kaggle/input/**/000_0000?.u16", recursive=True))
print("shard ditemukan:", len(shards))
for s in shards: print("  ", s, round(os.path.getsize(s) / 1e9, 2), "GB", "| indeks dokumen:", os.path.exists(s + ".docs.npy"))
assert len(shards) == EXPECTED_SHARDS, f"diharapkan {EXPECTED_SHARDS} shard, ditemukan {len(shards)}: pasang semua dataset shard (atau ubah EXPECTED_SHARDS dan konsisten di semua sesi)"
assert all(os.path.exists(s + ".docs.npy") for s in shards), "setiap shard butuh berkas .u16.docs.npy di sebelahnya"
last = shards[-1]
ndocs = int(np.load(last + ".docs.npy", mmap_mode="r").shape[0])
split = ndocs - VAL_DOCS
assert split > 0
TRAIN_SPEC = ",".join(shards[:-1] + [f"{last}@0:{split}"])
VAL_SPEC = f"{last}@{split}:"
print("data latih:", len(shards), "shard (shard terakhir dipotong di dokumen", split, "dari", ndocs, ") | validasi:", VAL_DOCS, "dokumen")
free = shutil.disk_usage("/kaggle/working").free / 1e9
print("ruang bebas /kaggle/working: %.1f GB (checkpoint 6,5 GB; butuh >= 14 GB saat menyimpan)" % free)
assert free > 14, "ruang /kaggle/working kurang"


In [ ]:
CK = f"{WORK}/ckpt" + ("_smoke" if SMOKE else ""); os.makedirs(CK, exist_ok=True)
if not SMOKE:
    shutil.rmtree(f"{WORK}/ckpt_smoke", ignore_errors=True)   # buang sisa uji asap (13 GB) agar disk cukup
def meta_of(f):
    m = re.sub(r"[.]pt$", ".meta.json", f)
    try: return json.load(open(m, encoding="utf-8"))
    except Exception: return None
cands = []
for f in glob.glob("/kaggle/input/**/gpt_last.pt", recursive=True) + glob.glob("/kaggle/input/**/gpt_step_*.pt", recursive=True) + glob.glob(f"{CK}/gpt_*.pt"):
    md = meta_of(f)
    if md and md.get("env") == "production_540m":
        cands.append((int(md["step"]), f))
    else:
        print("diabaikan (bukan checkpoint 540M atau tanpa .meta.json):", f)
cands.sort()
RESUME = cands[-1][1] if (cands and not SMOKE) else None
START_STEP = cands[-1][0] if RESUME else 0
for st, f in cands: print("  checkpoint:", st, f, "<- DIPAKAI" if f == RESUME else "")
if not SMOKE:
    if FIRST_SESSION:
        assert RESUME is None, "FIRST_SESSION=True tetapi ada checkpoint 540M di Input; set FIRST_SESSION=False (atau lepas dataset checkpoint)"
    else:
        assert RESUME, "FIRST_SESSION=False tetapi tidak ada checkpoint 540M ditemukan: pasang dataset checkpoint sesi sebelumnya (berisi gpt_last.pt dan gpt_last.meta.json)"
print("mulai dari step:", START_STEP, "/", MAX_STEPS, "| resume:", RESUME)


In [ ]:
if not SMOKE and START_STEP >= MAX_STEPS:
    print("SELESAI: MAX_STEPS tercapai; checkpoint akhir:", RESUME)
else:
    def step_of(f):
        m = meta_of(f); return int(m["step"]) if m else -1
    def prune_loop(stop):
        # simpan hanya 1 checkpoint periodik terbaru (6,5 GB per berkas; kuota output 20 GB)
        while not stop.is_set():
            steps = sorted(glob.glob(f"{CK}/gpt_step_*.pt"), key=step_of)
            for f in steps[:-1]:
                for g in (f, re.sub(r"[.]pt$", ".meta.json", f)):
                    try: os.remove(g)
                    except OSError: pass
            stop.wait(60)
    stop = threading.Event(); threading.Thread(target=prune_loop, args=(stop,), daemon=True).start()

    cmd = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2", "tools/train_gptV2.py",
           "--env", "production_540m", "--amp", "--grad-checkpoint", "--fused-adamw", "--seed", "1234",
           "--token-file", TRAIN_SPEC, "--val-token-file", VAL_SPEC, "--save-dir", CK, "--no-best-save",
           "--scheduler", "cosine", "--epochs", "1", "--batch-size", "2"]
    ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    if SMOKE:
        cmd += ["--max-steps", "3", "--grad-accum", "2", "--log-every", "1", "--save-every", "2", "--val-every", "3", "--val-steps", "3"]
    else:
        cmd += ["--max-steps", str(MAX_STEPS), "--grad-accum", "128", "--log-every", "5", "--save-every", str(SAVE_EVERY), "--val-every", str(SAVE_EVERY),
                "--val-steps", "50", "--time-limit-s", str(TIME_LIMIT_S)]
        if RESUME: cmd += ["--resume", RESUME]
    print(" ".join(cmd))
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV)
    def watchdog():
        if not stop.wait(HARD_KILL_S):
            print("[notebook] pengaman: proses tidak selesai, dihentikan paksa", flush=True); p.kill()
    threading.Thread(target=watchdog, daemon=True).start()
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True); lines.append(line)
    p.wait(); stop.set()
    print("lama (jam):", round((time.time() - t0) / 3600, 2), "| kode keluar:", p.returncode)
    if SMOKE:   # kecepatan dari langkah pertama ke terakhir (token per langkah = 2 GPU x batch 2 x akum 2 x 1024)
        rx = re.compile(r"step=(\d+) .*elapsed=([0-9.]+)s")
        pts = [(int(m.group(1)), float(m.group(2))) for m in (rx.search(l) for l in lines if l.startswith("epoch=")) if m]
        if len(pts) >= 2 and pts[-1][1] > pts[0][1]:
            (s1, t1), (s2, t2) = pts[0], pts[-1]
            tps = (s2 - s1) * 2 * 2 * 2 * 1024 / (t2 - t1)
            print("kecepatan terukur di data nyata: %.0f token/detik (bench: ±3.550); ±%.0f jam per 10 miliar token" % (tps, 1e10 / tps / 3600))
        assert p.returncode == 0, "UJI ASAP GAGAL: kirim 30 baris terakhir keluaran di atas"
        print("UJI ASAP BERHASIL. Set SMOKE=False untuk melatih.")
    else:
        last_pt = f"{CK}/gpt_last.pt"; md = meta_of(last_pt) if os.path.exists(last_pt) else None
        if md:   # bersihkan checkpoint periodik: output cukup berisi gpt_last.pt (6,5 GB)
            for f in glob.glob(f"{CK}/gpt_step_*"): os.remove(f)
            done = int(md["step"])
            print("SELESAI sesi: step", done, "/", MAX_STEPS, "= %.1f%%" % (100 * done / MAX_STEPS))
            if done < MAX_STEPS:
                per_session = max(1, done - START_STEP)
                print("sisa ±%d sesi serupa. Berikutnya: unggah output sebagai dataset checkpoint, FIRST_SESSION=False." % -(-(MAX_STEPS - done) // per_session))
        else:
            print("PERINGATAN: gpt_last.pt tidak ditulis; periksa log. Checkpoint periodik terakhir masih ada di", CK)


In [ ]:
print(subprocess.run(["ls", "-la", CK], capture_output=True, text=True).stdout)
print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)
